# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [22]:
import pandas as pd
import os

# connecting through google drive
from google.colab import drive
drive.mount('/content/drive')

# loading in the data
PATH = '/content/drive/MyDrive/Colab Notebooks/product_catalog.csv'
cat = pd.read_csv(PATH)

# print summary info
print("Rows:", len(cat))
print(cat.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rows: 45
             sku          product_name            category subcategory  \
0          PT-12  pop-tarts strawberry     Breakfast Foods         NaN   
1     SKU#459812   Strawberry Pop-Tart           Breakfast   Pop-Tarts   
2  POPTART-STRAW  Pop-Tarts Strawberry           Breakfast         NaN   
3  PT_STRAWBERRY  pop-tarts strawberry           breakfast   Pop-Tarts   
4         WTR-24    BOTTLED WATER 24PK  Emergency Supplies     Bottled   

   unit_cost           supplier package_size  
0       1.93  Johnson & Johnson       4-pack  
1       2.26           Roku Inc          4pk  
2       2.11           Sara Lee       6 pack  
3        NaN         Campbell's       4 pack  
4       2.64           Sara Lee       1 pack  


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [23]:
# TODO (hint: .str.lower().str.contains('pop'))

# filter on product that has 'pop' & case-insensitive by using str.lower()
pop_included = cat['product_name'].str.lower().str.contains('pop')

# matching rows
pop_rows = cat[pop_included]

# count the number of matching rows
pop_count = pop_included.sum()

print(pop_count)
print(pop_rows)

4
             sku          product_name         category subcategory  \
0          PT-12  pop-tarts strawberry  Breakfast Foods         NaN   
1     SKU#459812   Strawberry Pop-Tart        Breakfast   Pop-Tarts   
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast         NaN   
3  PT_STRAWBERRY  pop-tarts strawberry        breakfast   Pop-Tarts   

   unit_cost           supplier package_size  
0       1.93  Johnson & Johnson       4-pack  
1       2.26           Roku Inc          4pk  
2       2.11           Sara Lee       6 pack  
3        NaN         Campbell's       4 pack  


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [24]:
# TODO

# strawberry poptarts
strawberry_rows = pop_rows[pop_rows['product_name'].str.lower().str.contains('strawberry', na=False)]

# poptarts that are not strawberry
other_pop_rows = pop_rows[~pop_rows['product_name'].str.lower().str.contains('strawberry', na=False)]

# printing the strawberry poptarts & not strawberry
print(strawberry_rows)
print(other_pop_rows)

             sku          product_name         category subcategory  \
0          PT-12  pop-tarts strawberry  Breakfast Foods         NaN   
1     SKU#459812   Strawberry Pop-Tart        Breakfast   Pop-Tarts   
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast         NaN   
3  PT_STRAWBERRY  pop-tarts strawberry        breakfast   Pop-Tarts   

   unit_cost           supplier package_size  
0       1.93  Johnson & Johnson       4-pack  
1       2.26           Roku Inc          4pk  
2       2.11           Sara Lee       6 pack  
3        NaN         Campbell's       4 pack  
Empty DataFrame
Columns: [sku, product_name, category, subcategory, unit_cost, supplier, package_size]
Index: []


**Pop-Tart rows that are a different product:** None; there are no different Pop-Tarts. All Pop-Tart entries in the catalog seem to be strawberry-flavored, so no other flavors need to remain separate.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [25]:
# TODO

# creating a mapping - dictionary for each variant SKU and canonical SKU
sku_mapping = {
    'SKU#459812': 'PT-12',
    'POPTART-STRAW': 'PT-12',
    'PT_STRAWBERRY': 'PT-12'
}

# apply mapping - keeping unmapped SKU unchanged - not making them NAN
cat['canonical_SKU'] = cat['sku'].replace(sku_mapping)

# print the new mapping
cat[['sku', 'product_name', 'canonical_SKU']].head(10)

,sku,product_name,canonical_SKU
0,PT-12,pop-tarts strawberry,PT-12
1,SKU#459812,Strawberry Pop-Tart,PT-12
2,POPTART-STRAW,Pop-Tarts Strawberry,PT-12
3,PT_STRAWBERRY,pop-tarts strawberry,PT-12
4,WTR-24,BOTTLED WATER 24PK,WTR-24
5,WATER24PK,Bottled Water 24-Pack,WATER24PK
6,WTR_24PK,Water - 24 Pack,WTR_24PK
7,BATT-100,BATTERIES AA 4PK,BATT-100
8,BAT-AA4,AA Batteries 4-Pack,BAT-AA4
9,BATT_AA_4PK,AA Batteries 4-Pack,BATT_AA_4PK


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [26]:
# TODO

# printing distinct category values and counts (before)
print(cat['category'].value_counts())

# stripping whitespace and convert to title case with .strip() & .title() functions
cat['category'] = cat['category'].str.strip().str.title()

# printing distinct category values and countss (after normalization)
print(cat['category'].value_counts())

category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64
category
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health                 2
Breakfast Foods        1
Alcoho

**Note:** After we normalized the category names, some of the labels still appear to represent the same general category. Some of the examples are Breakfast/Breakfast Foods, Emergency/Emergency Supplies, Baby/Baby Products, Alcohol/Alcoholic Beverages, and Grocery/Groceries. They may represent the same category and need further review to determine if they need to be consolidated.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [27]:
# TODO

# import library
import string

# normalize each product name - lowercase, strip punctuation, sort
def normalize_name(name):
  name = str(name).lower().strip()
  name = name.translate(str.maketrans('', '', string.punctuation))
  return ' '.join(sorted(name.split()))

# create the normalized version for each of the product names
cat['normalized_name'] = cat['product_name'].apply(normalize_name)

# group the products by the normalized name - finding the groups with multiple SKUs
name_groups= cat.groupby('normalized_name')['sku'].agg(list)
duplicate_canidates = name_groups[name_groups.apply(lambda skus: len(set(skus)) > 1)]

# printing the duplicate canidates
print(duplicate_canidates)

normalized_name
24pack beer light                        [BEER-24, LTBEER24]
4pack aa batteries                    [BAT-AA4, BATT_AA_4PK]
board game monopoly        [BG-MONO, MONOPOLY-01, GAME-MONO]
bread white                           [BREAD-WH, WHITEBREAD]
butter peanut                            [PB-16, PNUTBTR-16]
candles emergency                     [CNDL-EM, CANDLE-EMRG]
canned soup                            [SOUP-CAN, CNSOUP-01]
cards playing                       [CARDS-01, PLAYCARD-STD]
chips pack variety                   [CHIPS-VP, CHIPVARIETY]
duty heavy tarp                         [TARP-HD, HDTARP-01]
flashlight led                         [FLASH-01, LED-FLASH]
paper towels                         [PTOWEL-6, PAPERTOWEL6]
poptarts strawberry    [PT-12, POPTART-STRAW, PT_STRAWBERRY]
Name: sku, dtype: object


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [28]:
# TODO

# checking the number of rows before applying the mapping
rows_before = len(cat)

# applying the mapping
cat['canonical_SKU'] = cat['sku'].replace(sku_mapping)

# check to make sure the mapping is safe
assert cat['canonical_SKU'].notna().all() # verifies every row has a canonical SKU
assert len(cat) == rows_before # verifies that the number of rows did not change

# print after the assertions are passed
print('Mapping is safe: no null canonical SKUs and no rows added or removed.')

Mapping is safe: no null canonical SKUs and no rows added or removed.


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

A) Two of the product families beyond Pop-Tarts that appear fragmented are Monopoly board games & white bread. Monopoly has three different SKU's, such as BG-MONO, MONOPOLY-01, and GAME-MONO. White bread has two different SKU's, such as BREAD-WH and WHITEBREAD. These two families may represent the same product even with different SKU identifiers, so we should look into their categorization.

B) Consolidation becomes a business decision when two products or categories have similar names but we are not sure whether they should actually be treated as the same thing. For example, the catalog contains both Emergency and Emergency Supplies. I would not automatically combine them because they may represent different business categories. I would ask a manager or someone on the merchandising team to confirm whether these should be consolidated or remain separate.